In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.layers import Dense,Conv2D,Flatten,MaxPooling2D
from keras import Sequential
from keras.datasets import mnist

In [2]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 17s 1us/step


In [3]:
x_train, x_test = x_train / 255.0, x_test / 255.0

In [4]:
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

In [5]:
model = Sequential()
model.add(Conv2D(8, kernel_size=(3,3),activation='relu', name = 'conv1', input_shape=(28,28,1)))
model.add(MaxPooling2D(pool_size=(2,2), name = 'pool1'))
model.add(Conv2D(16, kernel_size=(3, 3), activation='relu', name='conv2'))
model.add(MaxPooling2D(pool_size=(2, 2), name='pool2'))
model.add(Flatten(name='flatten'))
model.add(Dense(64, activation='relu', name='dense1'))
model.add(Dense(10, activation='softmax', name='output'))

c:\Users\Soham Manna\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [6]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1 (Conv2D)                  │ (None, 26, 26, 8)      │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 13, 13, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 11, 11, 16)     │         1,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 5, 5, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 64)             │        25,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,562 (107.66 KB)

 Trainable params: 27,562 (107.66 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [8]:
print("\n--- Starting Training ---")
history = model.fit(x_train, y_train, epochs=5, validation_split=0.1, batch_size=64)


--- Starting Training ---
Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.8145 - loss: 0.6432 - val_accuracy: 0.9763 - val_loss: 0.0845
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9721 - loss: 0.0905 - val_accuracy: 0.9807 - val_loss: 0.0664
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9802 - loss: 0.0631 - val_accuracy: 0.9853 - val_loss: 0.0488
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9846 - loss: 0.0494 - val_accuracy: 0.9842 - val_loss: 0.0564
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9866 - loss: 0.0431 - val_accuracy: 0.9860 - val_loss: 0.0469


In [9]:
print("\n--- Evaluating Model ---")
test_loss, test_acc = model.evaluate(x_test,  y_test, verbose=2)
print(f"\nFinal Test Accuracy: {test_acc*100:.2f}%")


--- Evaluating Model ---
313/313 - 1s - 2ms/step - accuracy: 0.9854 - loss: 0.0430

Final Test Accuracy: 98.54%


In [ ]:
import numpy as np
import tensorflow as tf

# Load the model you just trained (Make sure the filename matches what you saved)
model_name = 'mnist_hls_model.keras' # Change to .h5 if you saved it as .h5
try:
    model = tf.keras.models.load_model(model_name)
    print(f"Successfully loaded {model_name}")
except Exception as e:
    print(f"Error loading model: {e}")

# Open a new header file to write the C++ arrays
with open('weights.h', 'w') as f:
    f.write('// Auto-generated weights for HLS MNIST CNN\n')
    f.write('#pragma once\n\n')

    # Iterate through every layer in your model
    for layer in model.layers:
        # get_weights() returns a list: [weights_array, biases_array]
        layer_weights = layer.get_weights()

        if len(layer_weights) > 0:
            print(f"Extracting weights from layer: {layer.name}")

            # Extract and flatten the multi-dimensional arrays into 1D arrays for C++
            w = layer_weights[0].flatten()
            b = layer_weights[1].flatten()

            # 1. Write the Weights
            f.write(f'const float {layer.name}_weights[{len(w)}] = {{\n')
            # Join all numbers with a comma, format to 6 decimal places
            f.write(', '.join([f"{val:.6f}" for val in w]))
            f.write('\n};\n\n')

            # 2. Write the Biases
            f.write(f'const float {layer.name}_biases[{len(b)}] = {{\n')
            f.write(', '.join([f"{val:.6f}" for val in b]))
            f.write('\n};\n\n')

print("\nSuccess! Download 'weights.h' from your Colab files.")